# TF-IDF

Based on Term Frequency/Inverted Index last time

The text files are placed in folder './iitfidf', i.e. we have three files in the folder: 1.txt, 2.txt, 3.txt

The wholeTextFiles will read the files, and output a tuple (filename, fulltext)

data (i.e., wholeTextFiles) -> (filename, fulltext)

Here we calculate the tf "Term Frequency"

In [14]:
import re
def removeSomeTexts(text):
    return re.sub(r'[0-9a-z\s]','', text)

In [15]:
import jieba

from pyspark import SparkConf, SparkContext
import os, re
from pyspark.sql.session import SparkSession

spark = SparkSession.builder.master('local').appName('CN-MyTf-Idf').getOrCreate()
    
sc = spark.sparkContext
#sc = SparkContext.getOrCreate(SparkConf())
#sc._conf.set('spark.executor.memory', '8g')
#sc._conf.set('spark.driver.memory', '8g')
#sc._conf.set('spark.driver.maxResultsSize', '0')

data = sc.wholeTextFiles('data/cn_tfidf')

numFiles = data.count()
          
tf1 = data.flatMap(lambda x: [(os.path.basename(x[0]),removeSomeTexts(i.lower())) for i in re.split('\W+', x[1])])\
        .filter(lambda x: x[1] != '')

tf1.take(10)

[('sanguo.txt', '三国演义'),
 ('sanguo.txt', '第回'),
 ('sanguo.txt', '宴桃园豪杰三结义'),
 ('sanguo.txt', '斩黄巾英雄首立功'),
 ('sanguo.txt', '滚滚长江东逝水'),
 ('sanguo.txt', '浪花淘尽英雄'),
 ('sanguo.txt', '是非成败转头空'),
 ('sanguo.txt', '青山依旧在'),
 ('sanguo.txt', '几度夕阳红'),
 ('sanguo.txt', '白发渔樵江渚上')]

In [16]:
# ((filename, token), TF)
tf2 = tf1.flatMap(lambda x: [((i, x[0]),1) for i in jieba.cut(x[1])])\
        .reduceByKey(lambda a, b: a + b)\
        .map(lambda x: ((x[0][1],x[0][0]),x[1]))
tf2.take(10)

[(('sanguo.txt', '三国演义'), 1),
 (('sanguo.txt', '第回'), 70),
 (('sanguo.txt', '宴'), 16),
 (('sanguo.txt', '桃园'), 19),
 (('sanguo.txt', '豪杰'), 22),
 (('sanguo.txt', '三'), 110),
 (('sanguo.txt', '结义'), 14),
 (('sanguo.txt', '斩'), 369),
 (('sanguo.txt', '黄巾'), 40),
 (('sanguo.txt', '英雄'), 83)]

In [17]:
# tf: ((token, (filename, tf)) 
tf3 = tf1.flatMap(lambda x: [((i, x[0]),1) for i in jieba.cut(x[1])])\
        .reduceByKey(lambda a, b: a + b)\
        .map(lambda x: (x[0][0],(x[0][1],x[1])))
tf3.take(10)

[('三国演义', ('sanguo.txt', 1)),
 ('第回', ('sanguo.txt', 70)),
 ('宴', ('sanguo.txt', 16)),
 ('桃园', ('sanguo.txt', 19)),
 ('豪杰', ('sanguo.txt', 22)),
 ('三', ('sanguo.txt', 110)),
 ('结义', ('sanguo.txt', 14)),
 ('斩', ('sanguo.txt', 369)),
 ('黄巾', ('sanguo.txt', 40)),
 ('英雄', ('sanguo.txt', 83))]

In [18]:
# tf = 1+log10(wc)
import math
tf = tf3.map(lambda x: (x[0],(x[1][0], 1+math.log10((x[1][1]) ))))

tf.take(10)

[('三国演义', ('sanguo.txt', 1.0)),
 ('第回', ('sanguo.txt', 2.845098040014257)),
 ('宴', ('sanguo.txt', 2.204119982655925)),
 ('桃园', ('sanguo.txt', 2.278753600952829)),
 ('豪杰', ('sanguo.txt', 2.342422680822206)),
 ('三', ('sanguo.txt', 3.041392685158225)),
 ('结义', ('sanguo.txt', 2.146128035678238)),
 ('斩', ('sanguo.txt', 3.56702636615906)),
 ('黄巾', ('sanguo.txt', 2.6020599913279625)),
 ('英雄', ('sanguo.txt', 2.9190780923760737))]

## Computing Inverse Document Frequency (IDF)

The IDF score tells us how important is a particular word in the whole corpus. For instance, if a word is appearing in each document then the IDF score will be zero for that.

We know that the total number of documents for our case is 3. So, our next step will be to compute how many documents contain a particular token w. Further, we can use this information to calculate the IDF scores for all the tokens in the corpus.

We will be chaining the previous map/reduce computation to calculate IDF in this section. After the reduced step in the previous section, we obtained key/value pairs for respective tokens along with its term frequency.

In the first step, we will map the previous key/value pair to a new key/value pair. Here, the key will the token and its value will be the document id TF for that token along with a counter of 1. This one here indicates the presence of a word in document id associated with that.

 ((filename, token), TF) -> (token, (filename, TF, 1))

Then in the second map, we will extract the token and the number of counter of 1 representing its occurrence in certain documents.

(token, (filename, TF, 1)) -> (token, 1)

Then in the third step, we will reduce by key to obtain the count of documents containing a particular token w (wordcount).

(token, 1) -> (token, 1+1+1+...)

Now in the fourth step, since we have the number of documents containing each token w, we will just map this final output with the logarithmic transformation to compute the IDF score.

(token, doc-freq-containing-token) -> (token, IDF)

In [21]:

# (token, (filename, tf) -> (token, (filename, TF, 1))
idf = tf3.map(lambda x: (x[0], (x[1][0], x[1][1], 1)))
idf.take(10)

[('三国演义', ('sanguo.txt', 1, 1)),
 ('第回', ('sanguo.txt', 70, 1)),
 ('宴', ('sanguo.txt', 16, 1)),
 ('桃园', ('sanguo.txt', 19, 1)),
 ('豪杰', ('sanguo.txt', 22, 1)),
 ('三', ('sanguo.txt', 110, 1)),
 ('结义', ('sanguo.txt', 14, 1)),
 ('斩', ('sanguo.txt', 369, 1)),
 ('黄巾', ('sanguo.txt', 40, 1)),
 ('英雄', ('sanguo.txt', 83, 1))]

In [22]:
# (token, (filename, TF, 1)) -> (token, 1)
idf = idf.map(lambda x: (x[0], x[1][2]))
#idf.collect()
idf.take(10)

[('三国演义', 1),
 ('第回', 1),
 ('宴', 1),
 ('桃园', 1),
 ('豪杰', 1),
 ('三', 1),
 ('结义', 1),
 ('斩', 1),
 ('黄巾', 1),
 ('英雄', 1)]

In [23]:
# (token, 1) -> (token, 1+1+1+...)
idf = idf.reduceByKey(lambda x, y: x + y)
idf.take(10)

[('三国演义', 2),
 ('第回', 1),
 ('宴', 1),
 ('桃园', 1),
 ('豪杰', 1),
 ('三', 2),
 ('结义', 1),
 ('斩', 2),
 ('黄巾', 1),
 ('英雄', 2)]

In [24]:
# (token, doc-freq-containing-token) -> (token, IDF)
idf = idf.map(lambda x: (x[0], math.log10(numFiles / (x[1]))))
idf.take(10)

[('三国演义', 0.0),
 ('第回', 0.3010299956639812),
 ('宴', 0.3010299956639812),
 ('桃园', 0.3010299956639812),
 ('豪杰', 0.3010299956639812),
 ('三', 0.0),
 ('结义', 0.3010299956639812),
 ('斩', 0.0),
 ('黄巾', 0.3010299956639812),
 ('英雄', 0.0)]

## Computing TF-IDF

Now we have two RDDs one having the document id along with its document term frequency for each token. Another RDD having IDF scores for each token.

First, we will perform an inner join to assign each token with a document id, TF, and IDF score.

Second, we will map this function to multiply TF and IDF values of each token associated with respective document id.

Third, we sort the key

New Key/Value pair -> (filename, (token, TF-IDF))

In [25]:
tfidf = tf3.join(idf)\
            .map(lambda x: (x[1][0][0], (x[0], x[1][0][1] * x[1][1])))\
            .sortByKey()

tfidf.take(20)

[('sanguo.txt', ('三国演义', 0.0)),
 ('sanguo.txt', ('第回', 21.072099696478684)),
 ('sanguo.txt', ('桃园', 5.719569917615643)),
 ('sanguo.txt', ('豪杰', 6.622659904607587)),
 ('sanguo.txt', ('斩', 0.0)),
 ('sanguo.txt', ('英雄', 0.0)),
 ('sanguo.txt', ('首', 0.0)),
 ('sanguo.txt', ('立功', 6.622659904607587)),
 ('sanguo.txt', ('滚滚', 0.0)),
 ('sanguo.txt', ('长江', 0.0)),
 ('sanguo.txt', ('东', 0.0)),
 ('sanguo.txt', ('逝水', 0.3010299956639812)),
 ('sanguo.txt', ('浪花', 0.0)),
 ('sanguo.txt', ('转头', 0.0)),
 ('sanguo.txt', ('空', 0.0)),
 ('sanguo.txt', ('依旧', 0.0)),
 ('sanguo.txt', ('在', 0.0)),
 ('sanguo.txt', ('几度', 0.0)),
 ('sanguo.txt', ('惯', 0.6020599913279624)),
 ('sanguo.txt', ('看', 0.0))]

In [26]:
index = tfidf.map(lambda x:(x[1][0], (x[0], x[1][1]))).filter(lambda x: x[0]!='')
index.take(10)

[('三国演义', ('sanguo.txt', 0.0)),
 ('第回', ('sanguo.txt', 21.072099696478684)),
 ('桃园', ('sanguo.txt', 5.719569917615643)),
 ('豪杰', ('sanguo.txt', 6.622659904607587)),
 ('斩', ('sanguo.txt', 0.0)),
 ('英雄', ('sanguo.txt', 0.0)),
 ('首', ('sanguo.txt', 0.0)),
 ('立功', ('sanguo.txt', 6.622659904607587)),
 ('滚滚', ('sanguo.txt', 0.0)),
 ('长江', ('sanguo.txt', 0.0))]

In [27]:
index = index.sortByKey()
index.take(100)

[('à', ('santi.txt', 0.3010299956639812)),
 ('ε', ('santi.txt', 0.3010299956639812)),
 ('τ', ('santi.txt', 0.3010299956639812)),
 ('ⅰ', ('santi.txt', 0.9030899869919435)),
 ('ⅱ', ('santi.txt', 1.505149978319906)),
 ('ⅲ', ('santi.txt', 0.9030899869919435)),
 ('〇', ('santi.txt', 0.3010299956639812)),
 ('一', ('sanguo.txt', 0.0)),
 ('一', ('santi.txt', 0.0)),
 ('一一', ('sanguo.txt', 0.0)),
 ('一一', ('santi.txt', 0.0)),
 ('一一分', ('sanguo.txt', 0.6020599913279624)),
 ('一万', ('sanguo.txt', 9.632959861247398)),
 ('一万两', ('sanguo.txt', 0.3010299956639812)),
 ('一万个', ('santi.txt', 0.6020599913279624)),
 ('一万五千', ('sanguo.txt', 0.0)),
 ('一万五千', ('santi.txt', 0.0)),
 ('一万五千个', ('santi.txt', 0.3010299956639812)),
 ('一万余', ('sanguo.txt', 0.3010299956639812)),
 ('一万倍', ('santi.txt', 0.6020599913279624)),
 ('一万公里', ('santi.txt', 0.3010299956639812)),
 ('一万刀', ('sanguo.txt', 0.3010299956639812)),
 ('一万匹', ('sanguo.txt', 0.3010299956639812)),
 ('一万四千', ('santi.txt', 0.6020599913279624)),
 ('一万多', ('santi.t

In [13]:
#save tfidf index as parquet format in hdfs format
lines = index.map(lambda x: (x[0], x[1][0], x[1][1])).toDF()

# If you have already this directory, delete it first
# hdfs dfs -rm -r /hdfs/output/tfidf_index
lines.write.save('data/cn_tfidf_index')